In [1]:
import time
import statistics

import numpy as np
import pandas as pd
import networkx as nx

from scipy import stats

In [2]:
G = nx.read_edgelist(
    "../data/processed/orkut_largest_community.edgelist",
    nodetype=int
)

print(
    G.number_of_nodes(),
    G.number_of_edges()
)

4785 119891


In [3]:
source = list(G.nodes())[0]

print(source)

26681


In [4]:
def benchmark(func, runs=30):

    times = []

    for _ in range(runs):

        start = time.perf_counter()

        func()

        end = time.perf_counter()

        times.append(
            end - start
        )

    mean = statistics.mean(times)

    std = statistics.stdev(times)

    ci = stats.t.interval(
        0.95,
        len(times)-1,
        loc=mean,
        scale=stats.sem(times)
    )

    return {
        "mean": mean,
        "std": std,
        "ci_low": ci[0],
        "ci_high": ci[1]
    }

In [5]:
bfs_result = benchmark(
    lambda: list(
        nx.bfs_tree(
            G,
            source
        )
    )
)

bfs_result

{'mean': 0.25920761333351644,
 'std': 0.17702799317428516,
 'ci_low': np.float64(0.19310427430426735),
 'ci_high': np.float64(0.3253109523627655)}

In [6]:
dfs_result = benchmark(
    lambda: list(
        nx.dfs_tree(
            G,
            source
        )
    )
)

dfs_result

{'mean': 0.5687298266665797,
 'std': 0.27002546910270286,
 'ci_low': np.float64(0.46790065942264575),
 'ci_high': np.float64(0.6695589939105137)}

In [8]:
is_eulerian = nx.is_eulerian(G)

print(
    "Euleriano:",
    is_eulerian
)

Euleriano: False


In [9]:
for u, v in G.edges():
    G[u][v]["weight"] = 1

dijkstra_result = benchmark(
    lambda: nx.single_source_dijkstra_path_length(
        G,
        source
    )
)

dijkstra_result

{'mean': 5.251862503333663,
 'std': 3.195138579351253,
 'ci_low': np.float64(4.058778150011348),
 'ci_high': np.float64(6.444946856655978)}

In [10]:
bellman_result = benchmark(
    lambda: nx.single_source_bellman_ford_path_length(
        G,
        source
    )
)

bellman_result

{'mean': 1.3304114233333773,
 'std': 1.0754584093069708,
 'ci_low': np.float64(0.9288286534700498),
 'ci_high': np.float64(1.7319941931967047)}

In [11]:
tarjan_result = benchmark(
    lambda: list(
        nx.articulation_points(
            G
        )
    )
)

tarjan_result

{'mean': 1.1607269566662581,
 'std': 1.4876416527036231,
 'ci_low': np.float64(0.6052324342497633),
 'ci_high': np.float64(1.716221479082753)}

In [12]:
prim_result = benchmark(
    lambda: nx.minimum_spanning_tree(
        G,
        algorithm="prim"
    )
)

prim_result

{'mean': 1.4285766899999242,
 'std': 0.5925493363675969,
 'ci_low': np.float64(1.207315131468324),
 'ci_high': np.float64(1.6498382485315244)}

In [13]:
kruskal_result = benchmark(
    lambda: nx.minimum_spanning_tree(
        G,
        algorithm="kruskal"
    )
)

kruskal_result

{'mean': 0.49501585333358283,
 'std': 0.08811515288709143,
 'ci_low': np.float64(0.4621131145041647),
 'ci_high': np.float64(0.527918592163001)}

In [14]:
results = pd.DataFrame([
    ["BFS", bfs_result],
    ["DFS", dfs_result],
    ["Dijkstra", dijkstra_result],
    ["Bellman-Ford", bellman_result],
    ["Tarjan", tarjan_result],
    ["Prim", prim_result],
    ["Kruskal", kruskal_result]
])

results.columns = [
    "Algoritmo",
    "Resultados"
]

results

,Algoritmo,Resultados
0,BFS,"{'mean': 0.25920761333351644, 'std': 0.1770279..."
1,DFS,"{'mean': 0.5687298266665797, 'std': 0.27002546..."
2,Dijkstra,"{'mean': 5.251862503333663, 'std': 3.195138579..."
3,Bellman-Ford,"{'mean': 1.3304114233333773, 'std': 1.07545840..."
4,Tarjan,"{'mean': 1.1607269566662581, 'std': 1.48764165..."
5,Prim,"{'mean': 1.4285766899999242, 'std': 0.59254933..."
6,Kruskal,"{'mean': 0.49501585333358283, 'std': 0.0881151..."
